## 元論文

In [4]:
import numpy as np
import pandas as pd
from collections import defaultdict
import sys
import os

sys.path.append(os.path.abspath(".."))  # 1階層上を追加
import pulp_exc.scflp_v4 as scflp

In [5]:
instances = [
    {"I": 20,  "J": 20,  "p": 2, "r": 2},
    {"I": 20,  "J": 20,  "p": 3, "r": 2},
    {"I": 20,  "J": 20,  "p": 2, "r": 3},

    {"I": 40,  "J": 40,  "p": 2, "r": 2},
    {"I": 40,  "J": 40,  "p": 3, "r": 2},
    {"I": 40,  "J": 40,  "p": 2, "r": 3},

    {"I": 60,  "J": 60,  "p": 2, "r": 2},
    {"I": 60,  "J": 60,  "p": 3, "r": 2},
    {"I": 60,  "J": 60,  "p": 2, "r": 3},

    {"I": 80,  "J": 80,  "p": 2, "r": 2},
    {"I": 80,  "J": 80,  "p": 3, "r": 2},
    {"I": 80,  "J": 80,  "p": 2, "r": 3},

    {"I": 100, "J": 100, "p": 2, "r": 2},
    {"I": 100, "J": 100, "p": 3, "r": 2},
    {"I": 100, "J": 100, "p": 2, "r": 3},
]


In [ ]:
n_runs = 10
beta = 0.1
base_seed = 20250909   # 任意の固定値（再現用）

# Branch-and-Cut の既定（あなたが動かした設定に近い）
solver_kwargs = dict(
    max_nodes=1000,
    max_rounds_per_node=3,
    tol=1e-3,
    pulp_solver="CBC",
    node_selection="bestbound",   # "dfs" も可
    log_level="info",
    separation="approx",
    cut_policy="auto",
    lp_relax=True,                # LP緩和（推奨）
)

def make_seed(inst_id: int, run: int, base: int = base_seed) -> int:
    """インスタンスIDと試行番号に依存した再現可能シード"""
    return int(np.random.SeedSequence([base, inst_id, run]).generate_state(1)[0])

all_results = []

for inst_id, params in enumerate(instances, start=1):
    for run in range(n_runs):
        seed = make_seed(inst_id, run)
        data = scflp.make_random_instance(
            I=params["I"], J=params["J"], p=params["p"], r=params["r"],
            beta=beta, seed=seed
        )

        res = scflp.scflp_branch_and_cut(data, **solver_kwargs)

        # 1試行ぶんの記録
        row = {
            "instance_id": inst_id,
            "run": run,
            "I": params["I"], "J": params["J"], "p": params["p"], "r": params["r"],
            "beta": beta,
            # branch_and_cut の返り値に合わせる
            "theta": res.get("theta_best", None),
            "time_sec": res.get("time_sec", None),
            "status": res.get("status", None),
            "nodes_explored": res.get("nodes_explored", None),
            "gap_bound": res.get("gap_bound", None),
            "total_cuts": res.get("total_cuts", None),
            "total_bulge": res.get("total_bulge", None),
            "total_submod": res.get("total_submod", None),
            "x_best_sites": res.get("x_best", None)
        }
        all_results.append(row)

# DataFrame にまとめて CSV 保存
df = pd.DataFrame(all_results)
out = "scflp_results_all.csv"
df.to_csv(out, index=False, encoding="utf-8")
print(f"保存完了: {out}")

════════════════════════════════════════════════════════════════════════════════
🚀 Branch-and-Cut（論文 Algorithm 1）開始
════════════════════════════════════════════════════════════════════════════════
────────────────────────────────────────────────────────────────────────────────
➤ 🌿 ノード展開: depth=0, |fix1|=0, |fix0|=0
➤    · ノード(depth=0): θ̂=1.000000, L=0.519766, viol=4.802e-01
✂️  カット追加: SUBMOD
➤    · ノード(depth=0): θ̂=1.000000, L=0.473068, viol=5.269e-01
✂️  カット追加: SUBMOD
➤    · ノード(depth=0): θ̂=0.802172, L=0.601493, viol=2.007e-01
✂️  カット追加: BULGE
⚠️  このノードでのカット上限に到達。
➤ 🌱 分枝: j*=10 → 子ノード depth=1 を2つ追加
────────────────────────────────────────────────────────────────────────────────
➤ 🌿 ノード展開: depth=1, |fix1|=1, |fix0|=0
➤    · ノード(depth=1): θ̂=0.757366, L=0.561787, viol=1.956e-01
✂️  カット追加: BULGE
➤    · ノード(depth=1): θ̂=0.709143, L=0.548988, viol=1.602e-01
✂️  カット追加: BULGE
➤    · ノード(depth=1): θ̂=0.687687, L=0.500703, viol=1.870e-01
✂️  カット追加: BULGE
⚠️  このノードでのカット上限に到達。
➤ 🌱 分枝: j*=17 → 

## 豊島

In [ ]:
%run C:\Users\theko\allocate-probrem\mutaion_gda\func_v3.ipynb

In [ ]:
import numpy as np
import pandas as pd
import random, time

results = []
num_rows_columns = 50   # サイト配置グリッドの一辺
alpha = 0.0             # 固定
beta = 0.1
n_runs = 10             # 繰り返し回数
eta = 0.01       # 学習率

for inst_id, params in enumerate(instances, start=1):
    objs_relaxed, objs_binary, objs_ex, times = [], [], [], []

    for run in range(n_runs):
        seed = make_seed(inst_id, run)

        # 需要点の重みベクトル
        h_i = np.full(params["I"], 1 / params["I"])
        J_L = {0}
        J_F = {1}

        start = time.time()
        x_R, y_R, obj_relaxed, x_proj, y_proj, obj_binary, obj_ex, candidate_sites, demand_points, history = lgda_solver(
            params["I"], params["J"], num_rows_columns,
            params["p"], params["r"],
            alpha, beta, h_i, J_L, J_F,
            eta_x=eta, eta_y=eta,
            mu=.1,
            max_iter=300_000,
            tau_interval=15_000,
            return_history=True
        )
        elapsed = time.time() - start

        objs_relaxed.append(obj_relaxed)
        objs_binary.append(obj_binary)
        objs_ex.append(obj_ex)
        times.append(elapsed)
        print(f"  試行 {run+1}/{n_runs} 完了: obj_relaxed={obj_relaxed}, obj_binary={obj_binary}, obj_ex={obj_ex}, time_sec={elapsed}")

    # 各インスタンスの平均を保存
    row = {
        "I": params["I"],
        "J": params["J"],
        "p": params["p"],
        "r": params["r"],
        #"beta": beta,
        "obj_relaxed_avg": np.mean(objs_relaxed),
        "obj_binary_avg": np.mean(objs_binary),
        "obj_ex_avg": np.mean(objs_ex),
        "time_avg_sec": np.mean(times)
    }
    results.append(row)
    print(f"完了: instance_id={inst_id}, I={params['I']}, J={params['J']}, p={params['p']}, r={params['r']}")
    print(f"  obj_relaxed_avg={row['obj_relaxed_avg']}, obj_binary_avg={row['obj_binary_avg']}, obj_ex_avg={row['obj_ex_avg']}, time_avg_sec={row['time_avg_sec']}")

# DataFrame 化 & CSV 保存
df = pd.DataFrame(results)
df.to_csv("lgda_results_avg_nomask.csv", index=False, encoding="utf-8")
print(df)
print("保存完了: lgda_results_avg_nomask.csv")


3072965381
  試行 1/10 完了: obj_relaxed=0.6530308574431588, obj_binary=0.6190169665949756, obj_ex=0.6190169665949756, time_sec=130.16038465499878
2794667048
  試行 2/10 完了: obj_relaxed=0.5384343240767543, obj_binary=0.5524418460820405, obj_ex=0.5524418460820405, time_sec=131.60693836212158
1847930337
  試行 3/10 完了: obj_relaxed=0.6511618086987114, obj_binary=0.622324327535277, obj_ex=0.622324327535277, time_sec=168.67652916908264
2979255015
  試行 4/10 完了: obj_relaxed=0.6048536624079717, obj_binary=0.5508760371577603, obj_ex=0.5508760371577603, time_sec=154.92673206329346
406454889
  試行 5/10 完了: obj_relaxed=0.5991830342618353, obj_binary=0.5521536346810519, obj_ex=0.5521536346810519, time_sec=132.60926008224487
4112012237


KeyboardInterrupt: 